In [1]:
!pip install -q numpy scikit-learn sentence-transformers


In [ ]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records=records
        self.backend=backend
        texts=[r["text"] for r in records]
        if backend=="tfidf":
            self.vectorizer=TfidfVectorizer()
            self.matrix=self.vectorizer.fit_transform(texts)
        elif backend=="embedding":
            from sentence_transformers import SentenceTransformer
            self.model=SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix=self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            tokens=self.model.tokenizer(text, truncation=False)["input_ids"]
            if len(tokens)>self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self,query,k=3,filters=None):
        if not query.strip() or k<=0:
            return[]
        if self.backend=="tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz==0:
                return[]
            scores=(q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q=self.model.encode([query],normalize_embeddings=True)[0]
            scores=self.matrix @ q
        allowed=[
            i for i, r in enumerate(self.records)
            if all(r.get(key)==value for key, value in (filters or {}).items())
        ]
        ranked=sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return[(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record,score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [ ]:
%%writefile task2_chunks.py
import re
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):
    if size<=0 or overlap<0 or overlap>=size:
        raise ValueError("Invalid size or overlap")
    spans=word_spans(text)
    result=[]
    for i in range(0,len(spans),size-overlap):
        j=min(i+size,len(spans))
        result.append((offset+spans[i][0],offset+spans[j-1][1]))
        if j==len(spans):
            break
    return result
def sentence_chunks(text,size=40):
    if size<=0:
        raise ValueError("Size must be positive")
    cuts=[0]+[m.end() for m in re.finditer(r"[.!?]\s+",text)]+[len(text)]
    result,pending,count=[],None,0
    for a, b in zip(cuts,cuts[1:]):
        n=len(text[a:b].split())
        if not n:
            continue
        if n>size:
            if pending is not None:
                result.append(pending)
            result.extend(windows(text[a:b],size,0,a))
            pending,count=None,0
        elif pending is not None and count+n<=size:
            pending, count=(pending[0], b),count+n
        else:
            if pending is not None:
                result.append(pending)
            pending,count=(a,b),n
    if pending is not None:
        result.append(pending)
    return result
def make_chunks(docs,strategy):
    result=[]
    for doc_id,text in docs.items():
        spans=sentence_chunks(text) if strategy=="sentence"else windows(text, 40, int(strategy))
        for i,(a, b) in enumerate(spans):
            result.append({
                "id":f"{doc_id}:{i}", "doc": doc_id,
                "start":a, "end": b, "text": text[a:b]
            })
    return result
if __name__=="__main__":
    assert windows("")==[]
    assert windows("short text")==[(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap")
    long=" ".join(["word"] * 95)+"."
    assert all(len(long[a:b].split())<=40 for a, b in sentence_chunks(long))
    docs={"example": "Reset your password. Keep recovery codes private."}
    for strategy in["0", "10","sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"]==docs[c["doc"]][c["start"]:c["end"]]
            print(strategy,c)

Writing task2_chunks.py


In [ ]:
%%writefile task2_compare.py
from task2_chunks import make_chunks, windows, sentence_chunks

DOCS = {
    "late": (
        "Assignment Policy. Assignments course outline mein di gayi date "
        "ko raat 11:59 PM tak submit karni hoti hain. Late submission par "
        "har din ke hisaab se normally ten percent marks cut hote hain. "
        "Agar student bemar ho to extension ke liye deadline se pehle "
        "medical proof dena zaroori hai. Student ko medical certificate "
        "course coordinator ko bhejna hota hai aur written reply ka wait "
        "karna hota hai. Travel plans ya doosre courses ke workload ki "
        "wajah se extension nahi milti. Group projects mein bhi yehi rule "
        "follow hota hai. Har member ko apni illness ki information "
        "coordinator ko alag deni hoti hai. Late work submit hone par "
        "coordinator submission time check karta hai aur approved "
        "extension ke mutabiq penalty calculate karta hai. Student ko "
        "final marks ke saath penalty ki explanation bhi di jati hai. "
        "Student five working days ke andar penalty review ki request "
        "kar sakta hai. Tamam assignments check hone ke baad marks "
        "learning portal par upload kar diye jate hain."
    ),

    "password": (
        "IT Help Guide. Har student ko university ka ek account milta hai "
        "jo email, learning portal aur library ke liye use hota hai. "
        "Agar password bhool jayein to recovery link se password reset "
        "kar sakte hain. Ye link login page par hota hai aur registered "
        "phone number ya alternate email par bheja jata hai. Apne recovery "
        "codes kisi ke saath share na karein. Agar five wrong attempts "
        "ke baad account lock ho jaye to help desk se unlock karwana "
        "hota hai. Help desk student card aur problem ki short details "
        "mangta hai, lekin current password nahi mangta. Password reset "
        "hone ke baad shared computers se logout karein aur kam az kam "
        "twelve characters ka new password rakhein. University aur "
        "personal accounts ke liye same password use na karein. "
        "Passwords har one hundred and eighty days baad expire hote "
        "hain aur expiry se fourteen days pehle reminder milta hai."
    ),

    "assignment": (
        "Submission Guide. Har course mein written assignments submit "
        "karne ka same method hota hai taake files lost na hon. "
        "Assignments learning portal ke through upload karein. Final "
        "file PDF format mein honi chahiye aur file name mein course "
        "code aur student number likhna zaroori hai. Agar student portal "
        "access nahi kar sakta to deadline se pehle file coordinator "
        "ko email kar sakta hai aur sent email ka proof save rakhna "
        "chahiye. Portal twenty megabytes tak ki files accept karta hai. "
        "Bari files ko compress ya do parts mein divide karna chahiye. "
        "Har upload ke baad receipt milti hai jisme date aur time hota "
        "hai. Is receipt ko final marks aane tak save rakhein. Deadline "
        "se pehle latest uploaded file check hoti hai. Student zaroorat "
        "ke mutabiq file dobara upload kar sakta hai. Source code, data "
        "files aur images ko ek folder mein rakh kar report ke saath "
        "single archive mein submit karna hota hai. Har upload ka "
        "plagiarism check automatically hota hai."
    )
}

PAIRS = {
    "late": (
        "Late submission par har din ke hisaab se normally ten percent marks cut hote hain.",
        "Extension ke liye deadline se pehle medical proof dena zaroori hai."
    ),
    "password": (
        "Password bhool jayein to recovery link se password reset kar sakte hain.",
        "Five wrong attempts ke baad account lock ho jaye to help desk se unlock karwana hota hai."
    ),
    "assignment": (
        "Assignments learning portal ke through upload karein.",
        "Agar student portal access nahi kar sakta to deadline se pehle file coordinator ko email kar sakta hai"
    )
}
if __name__=="__main__":
    original_words={d: len(t.split()) for d, t in DOCS.items()}
    for d, t in DOCS.items():
        longest=max(len(s.split()) for s in t.replace("?", ".").split("."))
        print(d,"words:",original_words[d],
              "| longest sentence:", longest)
        assert d in PAIRS and all(p in t for p in PAIRS[d])
    total=sum(original_words.values())
    print()
    methods=[
        ("0","window 40,overlap 0"),
        ("10","window 40,overlap 10"),
        ("sentence","sentence-aware 40")
    ]
    for strategy,name in methods:
        chunks=make_chunks(DOCS, strategy)
        sizes=[len(c["text"].split()) for c in chunks]
        duplicate=sum(sizes) - total
        survived=sum(
            any(c["doc"] == d and rule in c["text"] and exc in c["text"]
                for c in chunks)
            for d,(rule, exc) in PAIRS.items()
        )
        print(f"{name}: chunks={len(chunks)}, "
              f"avg words={sum(sizes)/len(sizes):.1f}, "
              f"min={min(sizes)}, max={max(sizes)}, "
              f"duplicate words={duplicate}, "
              f"rule+exception={survived}/{len(PAIRS)}")
        for c in chunks:
            assert c["text"]==DOCS[c["doc"]][c["start"]:c["end"]]
            print(c["id"],c["start"], c["end"],
                  len(c["text"].split()), "words")
        print()
    print("Boundary tests")
    print("empty:",windows(""),sentence_chunks(""))
    print("short:",windows("short text"),sentence_chunks("short text"))
    try:
        windows("a b",2,2)
    except ValueError as error:
        print("Invalid overlap:",error)
    long = " ".join(["word"] * 95) + "."
    spans = sentence_chunks(long)
    print("95-word sentence:",len(spans),"chunks",
          [len(long[a:b].split()) for a,b in spans])

Writing task2_compare.py


In [5]:
!ls

lab6_search.py	sample_data  task2_chunks.py  task2_compare.py


In [ ]:
%%writefile create_dataset.py
import json
from pathlib import Path
from task2_compare import DOCS
docs = dict(DOCS)
docs["contractor"]=(
    "Remote Work Policy. Ye policy batati hai ke staff aur contractors "
    "kahan se kaam kar sakte hain aur unki requests kaise approve hoti "
    "hain. Employees apne manager ki permission se har week three days "
    "ghar se kaam kar sakte hain. Contractors sirf tab remote work kar "
    "sakte hain jab unke signed contract mein iski permission ho. "
    "Employee rules contractors par apply nahi hote. Is liye manager "
    "sirf zubani permission de kar contractor ko remote work allow nahi "
    "kar sakta. Agar contract mein remote work ka zikr nahi hai to "
    "contractor ko office se kaam karna hoga jab tak human resources "
    "office contract mein written change na kar de. Policy ka version "
    "one ab cancel ho chuka hai. Is purani policy mein contractors ko "
    "bina contract permission ke two days remote work ki ijazat thi. "
    "Purani policy ki copy ab use nahi karni chahiye. Contract change "
    "ki request human resources office ko contract number ke saath bheji "
    "jati hai. Office ten working days mein reply deta hai. Remote "
    "workers ko office hours mein available rehna hota hai, approved "
    "company laptops use karne hote hain aur secure network ke through "
    "connect rehna zaroori hai."
)

docs["printer"]=(
    "Printer Troubleshooting Guide. Ye guide office ke har floor par "
    "available shared printers aur unke common problems ke bare mein "
    "hai. Agar printer blank pages de raha ho to ink level check karein "
    "aur cleaning utility run karein. Iske baad test page print karein "
    "taake confirm ho sake ke problem solve hui hai ya nahi. Result "
    "maintenance log mein bhi likhein. Agar test page phir bhi blank "
    "ho to cartridge nikalein, protective tape check karein aur "
    "cartridge ko dobara properly lagayein. Agar scanner image blank "
    "aa rahi ho to ye alag problem hai. Is case mein document ki "
    "placement check karein aur correct scanning source select karein. "
    "Storage room ke purane laser printers mein drum unit use hota hai. "
    "Un models mein blank pages aane par drum replace karna hota hai, "
    "lekin ye rule current inkjet printers par apply nahi hota. "
    "Printing ke waqt printer ko kabhi open na karein. Paper jam hone "
    "par sheet nikalne se pehle printer ko off karein. Agar problem "
    "bar bar aaye to printer label aur failure time ke saath help desk "
    "ko report karein."
)

evidence={
    "late": "Illness extensions require medical evidence before the deadline.",
    "password": "Reset a forgotten password using the recovery link.",
    "assignment": "Upload assignments through the learning portal.",
    "contractor": "Contractors can work remotely only if the signed contract permits it.",
    "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
}

questions=[
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("What are tomorrow's flight prices?", None, "test"),
    ("Who won the football match?", None, "test")
]

queries=[
    dict(query=q,doc=doc,
         evidence=evidence[doc] if doc else None, split=split)
    for q, doc, split in questions
]
for d, e in evidence.items():
    assert e in docs[d], d
    print(d,len(docs[d].split()), "words")
for name, data in [("corpus.json", docs),("queries.json", queries)]:
    if Path(name).exists():
        raise FileExistsError("Refusing to overwrite " + name)
    Path(name).write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
print("Created corpus.json and queries.json")

Writing create_dataset.py


In [7]:
!python create_dataset.py

late 173 words
password 160 words
assignment 165 words
contractor 171 words
printer 176 words
Created corpus.json and queries.json


In [ ]:
%%writefile task3_retrieval.py
import csv, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks
def evaluate(scores,chunks,items):
    hits,reciprocal=[],[]
    for row,q in zip(scores,items):
        if q["doc"] is None:
            continue
        top=np.argsort(-row, kind="stable")[:3]
        good=[r for r,i in enumerate(top, 1)
                if chunks[i]["doc"]==q["doc"]
                and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1/good[0] if good else 0)
    if not hits:
        return None,None
    return float(np.mean(hits)),float(np.mean(reciprocal))
def main():
    with open("corpus.json",encoding="utf-8") as f:
        docs=json.load(f)
    with open("queries.json",encoding="utf-8") as f:
        items=json.load(f)
    assert docs and items
    for q in items:
        assert q["split"] in {"dev", "test"}
        if q["doc"] is not None:
            assert q["evidence"] in docs[q["doc"]]
    t=time.perf_counter()
    model=SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    print("Model load seconds:", time.perf_counter() - t)
    queries=[q["query"] for q in items]
    logs=[]
    for strategy in ["0","10","sentence"]:
        chunks=make_chunks(docs,strategy)
        texts=[c["text"] for c in chunks]
        lengths=[
            len(model.tokenizer(t,truncation=False)["input_ids"])
            for t in texts+queries
        ]
        print("Strategy:", strategy, "Chunks:", len(chunks),
              "Max tokens:",max(lengths))
        if max(lengths)>model.max_seq_length:
            raise ValueError("Model token limit exceeded")
        for method in ["tfidf", "embedding"]:
            t=time.perf_counter()
            if method=="tfidf":
                vectorizer=TfidfVectorizer()
                C=vectorizer.fit_transform(texts)
                Q=vectorizer.transform(queries)
            else:
                C=model.encode(texts, normalize_embeddings=True)
                Q=model.encode(queries, normalize_embeddings=True)
            encode_time = time.perf_counter() - t
            t=time.perf_counter()
            scores=(Q @ C.T).toarray() if method=="tfidf" else Q @ C.T
            search_time=time.perf_counter()-t
            for split in ["dev","test"]:
                indices=[i for i,q in enumerate(items)
                           if q["split"]==split]
                result=evaluate(
                    scores[indices], chunks, [items[i] for i in indices]
                )
                print(strategy,method,split,"Hit/MRR:",result,
                      "Encode/Search:",encode_time,search_time)
            for row,q in zip(scores,items):
                top=np.argsort(-row, kind="stable")[:3]
                for rank,i in enumerate(top, 1):
                    c=chunks[i]
                    record={
                        "strategy": strategy,"method": method,
                        "query": q["query"],"split":q["split"],
                        "rank":rank,"score":float(row[i]),
                        "chunk_id":c["id"],"doc": c["doc"],
                        "start": c["start"],"end":c["end"],
                        "text":c["text"],
                        "answerable":q["doc"] is not None
                    }
                    logs.append(record)
                    print(strategy, method,repr(q["query"]),
                          rank, f"{row[i]:.3f}",c["id"],
                          c["start"],c["end"],repr(c["text"][:70]))
    with open("retrieval_log.csv","w",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f, fieldnames=list(logs[0]))
        writer.writeheader()
        writer.writerows(logs)
    print("Saved retrieval_log.csv:",len(logs), "rows")
if __name__=="__main__":
    main()

Writing task3_retrieval.py


In [9]:
!python task3_retrieval.py

modules.json: 100% 349/349 [00:00<00:00, 1.44MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 626kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 24.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 332kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.49MB/s]

model.safetensors: downloading bytes:  60% 54.6M/90.9M [00:00<00:00, 105MB/s, 1.71MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 75.6MB/s, 8.20MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 80.9MB/s, 8.87MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6653.32it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 2.00MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.3MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 37.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 567kB/s]
config.json: 100% 190/190 [00:00<00:00, 758kB/s]
Model load seconds 3.924421076000044
Strategy 0 | chunks: 24 | max model tokens: 50 | limit: 256
0 tfid

In [ ]:
%%writefile document_chatbot.py
import json
from task2_chunks import make_chunks
from lab6_search import Retriever
if __name__ == "__main__":
    with open("corpus.json",encoding="utf-8") as f:
        docs=json.load(f)
    chunks=make_chunks(docs,"sentence")
    engine=Retriever(chunks,"embedding")
    print("Evidence assistant. Type exit to stop.")
    while True:
        try:
            query=input("Question: ")
        except(EOFError, KeyboardInterrupt):
            break
        if query.strip().lower()=="exit":
            break
        hits=engine.search(query)
        if not hits:
            print("No evidence found. Try again or contact a human.")
            continue
        print("Possible evidence (check relevance):")
        for i,(record, score) in enumerate(hits, 1):
            print(i, record["id"],f"{score:.3f}",
                  "characters",record["start"], record["end"])
        choice=input("Choose 1-3 to inspect (Enter to skip): ")
        if not choice.isdigit() or not 1<=int(choice)<=len(hits):
            print("No answer selected. Try again or contact a human.")
            continue
        record,score=hits[int(choice) - 1]
        print("Retrieved passage:",record["text"])
        print("Source:",record["doc"],
              "characters",record["start"],record["end"])

Writing document_chatbot.py


In [11]:
%run document_chatbot.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evidence assistant. Type exit to stop.
Question: How can I recover my login access?
Candidate evidence; relevance must be checked:
1 password:4 0.524 characters 614 830
2 password:2 0.503 characters 365 476
3 password:0 0.490 characters 0 174
Choose 1-3 to inspect; Enter if none apply: What are tomorrow's flight prices?
No answer selected. Rephrase or contact a human.
Question: What are tomorrow's flight prices?
Candidate evidence; relevance must be checked:
1 password:5 0.074 characters 830 942
2 late:0 0.069 characters 0 244
3 late:4 0.066 characters 813 897
Choose 1-3 to inspect; Enter if none apply: 2
Retrieved passage: Assessment Policy. Assignments are due at 11:59 PM on the date stated in the course outline. Late submission is normally penalized by ten percent for each day after the deadline. Illness extensions require medical evidence before the deadline. 
Source: late characters 0 244


In [14]:
from google.colab import files
files.download("corpus.json")
files.download("queries.json")
files.download("retrieval_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>